# Gold — `gold_resumo_cliente_mes`

## Bibliotecas

In [0]:
from pyspark.sql import functions as F

## Variáveis

In [0]:
# origem
gold_transacoes   = "novarota.gold.gold_transacoes_cartao"   # tem que rodar antes desse notebook
silver_contas     = "novarota.silver.contas"                 # pra saber desde quando o cliente existe
silver_transacoes = "novarota.silver.transacoes"
silver_estornos   = "novarota.silver.estornos"
silver_controle   = "novarota.silver.controle_cargas"

# destino
tabela_gold     = "novarota.gold.gold_resumo_cliente_mes"
tabela_controle = "novarota.gold.controle_gold"
nome_tabela     = "gold_resumo_cliente_mes"

# chave da tabela: uma linha por cliente por mês
chaves = ["id_cliente", "mes"]

print(f"Tabela alvo: {tabela_gold}")

## Ultima carga

In [0]:
# cargas que a silver terminou. O estornos é o último notebook da silver (e é ele que marca as estornadas)
cargas_silver = [
    l.data_referencia
    for l in spark.table(silver_controle).filter("tabela = 'estornos'").select("data_referencia").distinct().collect()
]

# cargas que essa tabela da gold já processou
if spark.catalog.tableExists(tabela_controle):
    cargas_processadas = [
        l.data_referencia
        for l in spark.table(tabela_controle).filter(f"tabela = '{nome_tabela}'").select("data_referencia").distinct().collect()
    ]
else:
    cargas_processadas = []

cargas_pendentes = sorted(c for c in cargas_silver if c not in cargas_processadas)

print(f"Cargas já processadas: {sorted(cargas_processadas)}")
print(f"Cargas pendentes: {cargas_pendentes}")

if not cargas_pendentes:
    dbutils.notebook.exit(f"Nada pendente pra {nome_tabela}")

## Meses afetados

In [0]:
cargas_sql = ", ".join(f"'{c}'" for c in cargas_pendentes)

query_meses = f"""
with tocados as (
-- meses das transações que chegaram nessas cargas (inclusive as atrasadas)
select trunc(data_transacao, 'MM') as mes
from {silver_transacoes}
where data_referencia in ({cargas_sql})

union

-- meses das transações que receberam estorno nessas cargas
select trunc(t.data_transacao, 'MM') as mes
from {silver_estornos} e
join {silver_transacoes} t on e.id_transacao = t.id_transacao
where e.data_referencia in ({cargas_sql})
)
,
ultimo_mes as (select trunc(max(data_transacao), 'MM') as mes from {silver_transacoes})


select mes from tocados

union

-- o mês seguinte também: a comparação dele com o mês anterior mudou
select add_months(t.mes, 1) as mes
from tocados t, ultimo_mes u
where add_months(t.mes, 1) <= u.mes
"""

meses = sorted(str(l.mes) for l in spark.sql(query_meses).collect())
meses_sql = ", ".join(f"'{m}'" for m in meses)

# primeiro e último mês afetado: limitam o que a query lê
inicio = meses[0]
fim = meses[-1]

print(f"Meses a recalcular: {meses}")

## Transformações

In [0]:
query = f"""
with primeiro_mes as (select min(mes) as mes from {gold_transacoes})
,
meses as (
-- do mês anterior ao primeiro afetado até o último. Esse mês a mais só serve de base pro lag
select explode(sequence(add_months(date'{inicio}', -1), date'{fim}', interval 1 month)) as mes)
,
clientes as (
-- o cliente entra no resumo a partir do mês em que abriu a primeira conta
select id_cliente, trunc(min(data_abertura), 'MM') as mes_inicio
from {silver_contas}
group by id_cliente)
,
clientes_meses as (
-- o esqueleto: uma linha por cliente por mês, tendo movimento ou não
select c.id_cliente, m.mes
from clientes c
join meses m        on m.mes >= c.mes_inicio
join primeiro_mes p on m.mes >= p.mes)
,
movimento as (
-- estornadas ficam fora do valor_total, mas aparecem nas colunas de estorno
select
    id_cliente,
    mes,
    count_if(not fl_estornada)                            as qtd_transacoes,
    sum(case when not fl_estornada then valor else 0 end) as valor_total,
    count_if(fl_estornada)                                as qtd_estornadas,
    sum(case when fl_estornada then valor else 0 end)     as valor_estornado
from {gold_transacoes}
where mes between add_months(date'{inicio}', -1) and date'{fim}'
group by id_cliente, mes)
,
completo as (
select
    cm.id_cliente,
    cm.mes,
    coalesce(mv.qtd_transacoes, 0)  as qtd_transacoes,
    coalesce(mv.valor_total, 0)     as valor_total,
    coalesce(mv.qtd_estornadas, 0)  as qtd_estornadas,
    coalesce(mv.valor_estornado, 0) as valor_estornado
from clientes_meses cm
left join movimento mv on cm.id_cliente = mv.id_cliente and cm.mes = mv.mes)
,
com_anterior as (
select
    *,
    lag(valor_total) over (partition by id_cliente order by mes) as valor_mes_anterior
from completo)


select
    id_cliente,
    mes,
    qtd_transacoes,
    valor_total,
    round(valor_total / nullif(qtd_transacoes, 0), 2)                                  as ticket_medio,
    qtd_estornadas,
    valor_estornado,
    valor_mes_anterior,
    round((valor_total - valor_mes_anterior) / nullif(valor_mes_anterior, 0) * 100, 2) as variacao_pct
from com_anterior
where mes in ({meses_sql})
"""

df = spark.sql(query)

print(f"Linhas dos meses afetados: {df.count()}")

## Conferência

In [0]:
dups_no_lote = (
    df.groupBy(*chaves).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Cliente/mês duplicados no lote (esperado 0): {dups_no_lote}")

df.orderBy("id_cliente", "mes").limit(20).display()

## Gravação por mês

In [0]:
if not spark.catalog.tableExists(tabela_gold):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_gold)
    )
    print(f"Tabela criada (carga inicial): {tabela_gold} | {df.count()} linhas")
else:
    # substitui só os meses afetados, o resto da tabela não é tocado
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"mes in ({meses_sql})")
        .saveAsTable(tabela_gold)
    )
    print(f"Meses regravados em {tabela_gold}: {meses}")

## Controle

In [0]:
# registra as cargas que essa tabela processou. Fica por último: só conta se a gravação deu certo
controle = (
    spark.createDataFrame([(nome_tabela, c) for c in cargas_pendentes], "tabela string, data_referencia string")
    .withColumn("processado_em", F.current_timestamp())
)
controle.write.format("delta").mode("append").saveAsTable(tabela_controle)

controle.display()

## Conferência pós-escrita

In [0]:
print(f"Conferência da tabela: {tabela_gold}")

df_gold = spark.table(tabela_gold)
print(f"Total de linhas: {df_gold.count()}")

dups = (
    df_gold.groupBy(*chaves).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Cliente/mês duplicados (esperado 0): {dups}")

# um cliente de cada cenário: mês zerado (CLI0010), cartão cancelado (CLI0025) e conta aberta no meio do ano (CLI0051)
df_gold.filter("id_cliente in ('CLI0010', 'CLI0025', 'CLI0051')").orderBy("id_cliente", "mes").display()